Setup vLLM on Kaggle GPU for Hermes Agent
This guide provides detailed instructions for deploying a vLLM server on Kaggle's free T4 GPU and exposing it via Cloudflare Tunnel for use with Hermes Agent.
Prerequisites
·	Kaggle account
·	Basic familiarity with Jupyter notebooks
·	Cloudflare account (for tunnel, though CLI install handles it)
Step-by-Step Setup
1. Create New Kaggle Notebook
1.	Go to kaggle.com
2.	Select "New Notebook"
3.	Choose "Notebook" environment
4.	Set GPU acceleration to T4 x2 (Settings → Accelerator)
Important: If T4 doesn't initialize:
·	Settings → Accelerator None → Save
·	Wait 10 seconds
·	GPU T4 x2 → Save
·	If still not working: Stop Session and start new notebook

2. Install Dependencies with Matching CUDA Versions (Cell 1)

In [ ]:
 # Check current PyTorch and CUDA versions
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

# Install PyTorch and TorchAudio with matching CUDA version (CUDA 13.0 for Kaggle T4)
# This ensures compatibility between PyTorch and TorchAudio (avoiding CUDA version mismatch)
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu130

# Then install vLLM and other dependencies
!pip install -q vllm transformers cloudflare
print("✓ Installed: torch, torchaudio, vllm, transformers, cloudflare")

3. Load Model and Tokenizer (Cell 2)

In [ ]:
from transformers import AutoTokenizer
from vllm import LLM

model_id = "Qwen/Qwen2-1.5B-Instruct"
print("Loading model (first time: ~1-2 min)...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
print("✓ Tokenizer loaded")

llm = LLM(
    model_id, 
    gpu_memory_utilization=0.7, 
    dtype="auto", 
    max_num_seqs=1
)
print("✓ Model loaded (1.5B, ~3GB VRAM)")

# Free memory for server
del llm


4. Start vLLM OpenAI-Compatible Server (Cell 3)

In [ ]:
import subprocess
import sys
import time

server_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2-1.5B-Instruct",
    "--host", "127.0.0.1",
    "--port", "8000",
    "--gpu-memory-utilization", "0.7",
    "--dtype", "auto",
    "--max-model-len", "4096",
    "--max-num-seqs", "1"
]

# Start server in background
server_proc = subprocess.Popen(
    server_cmd, 
    stdout=subprocess.PIPE, 
    stderr=subprocess.STDOUT
)

# Wait for server to initialize
time.sleep(10)
print("✓ vLLM running on 127.0.0.1:8000")

5. Install and Start Cloudflare Tunnel (Cell 4)

In [ ]:
import subprocess
import time
import re

print("Installing Cloudflare CLI...")
# Install cloudflared (the official CLI) using the install script
subprocess.run(
    ["bash", "-c", "curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o /tmp/cloudflared.deb"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
subprocess.run(["dpkg", "-i", "/tmp/cloudflared.deb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# Verify installation
subprocess.run(["cloudflared", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Starting tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

tunnel_url = None
timeout_start = time.time()

while time.time() - timeout_start < 60:
    line = tunnel_proc.stdout.readline().strip()
    if not line:
        time.sleep(0.5)
        continue
        
    print(f"[tunnel] {line}")
    
    if "https://" in line and "trycloudflare.com" in line:
        urls = re.findall(r"https://[^\\s]+trycloudflare\\.com[^\\s]*", line)
        if urls:
            tunnel_url = urls[0].rstrip(".,")
            break

if tunnel_url:
    print()
    print("="*70)
    print("✓ TUNNEL READY")
    print("="*70)
    print(f"API URL: {tunnel_url}")
    print()
    print("COPY THIS ON YOUR LOCAL MACHINE:")
    print(f'export LLM_API_URL="{tunnel_url}"')
    print("="*70)
else:
    print("✗ Could not get tunnel URL")
    print("Check tunnel logs above or restart tunnel cell")

### 6. Save Tunnel URL for Later Use

In [ ]:
After successful tunnel creation, save the URL:
```bash
# On your local machine (not in Kaggle notebook)
echo "export LLM_API_URL=\"https://your-actual-tunnel-id.trycloudflare.com\"" >> ~/.bashrc
source ~/.bashrc
Verification
Local Test (from your machine)
cd ~/llm-hermes
source venv-genai/bin/activate  # or create/use your Hermes environment

python agent_client_remote.py "Hello" --api-url $LLM_API_URL
Expected output: A coherent response from Qwen2-1.5B-Instruct.
Hermes Agent Configuration
Edit ~/.hermes/config.yaml:
llm:
  backend: "openai-compatible"
  api_url: "https://your-actual-tunnel-id.trycloudflare.com"  # <-- REPLACE PLACEHOLDER
  model: "Qwen/Qwen2-1.5B-Instruct"
  max_tokens_input: 4096
  max_tokens_output: 512
  timeout_s: 120
Then test: